<div style="padding: 20px; background-color: #000080; border-radius: 10px; box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);">
    <div style="border: 2px solid #000080; padding: 20px; text-align: center; border-radius: 10px; background-color: #ffffff;">
        <h1 style="color: #ffff00; font-size: 32px; text-transform: uppercase; letter-spacing: 2px; margin-bottom: 20px;">Easy EDA & modelling with BlueCast</h1>
        <div><em>
       If you like the content please consider an upvote. It is a great motivator to keep sharing code and ideas.
        Thank you!!!
    </em></div>
</div>

<h1 style="background-color: #000080; color: #ffff00;">Introducing BlueCast</h1>

Bluecast is an automl framework that offers a lightweight library with EDA, automl and xperiment tracking capabilities.
It offers many options for customization. Check out the repo to see many examples:
https://github.com/ThomasMeissnerDS/BlueCast

<h1 style="background-color: #000080; color: #ffff00;">Table of contents</h1>

* [Load the data](#1)
* [Feature engineering](#2)
    * [Add running features](#2.1)
* [EDA with BlueCast](#3)
    * [Target class distribution](#3.1)
    * [Feature type detection](#3.2)
    * [Univariate plots](#3.3)    
    * [Bivariate plots](#3.4) 
    * [Count pairs](#3.5)
    * [Correlation to target](#3.6)  
    * [Correlation heatmap](#3.7) 
    * [Mutual informtion score](#3.8)
    * [Dimensionality reduction using PCA](#3.9) 
    * [Show cumulative variance of principal components](#3.10)
    * [Dimensionality reduction using t-SNE](#3.11) 
    * [Map of associations between categorical features](#3.12)
    * [Missing values](#3.13)
    * [Do we have columns with high cardinality?](#3.14) 
* [Leakage detection](#4)
    * [Leakage detection for numerical columns](#4.1)
    * [Leakage detection for categorical columns](#4.2)  
* [Check for data drift](#5)
    * [Data drift for numerical columns](#5.1)
    * [Data drift for categorical columns](#5.2)
* [Building the pipeline in a few lines of code](#6)
* [Plot decision trees](#7)
* [Predict on new data](#8) 
* [Accessing the inbuilt experiment tracker](#9) 
    * [Understand most impactful parameters across all hyperparameters tests and model trainings](#9.1)
    * [Don't lose your progress!](#9.2) 
* [Submission time](#10)

In [ ]:
from IPython.core.display import HTML

# Define custom CSS directly in Python variable
custom_css = """
<style>
  :root {
    --header1_color: #204709;
    --header2_color: #42841F;
    --header3_color: #6EAF4B;
    --keyword_color: #cc241d; /* import */
    --string_color: #79740e;
    --number_color: #b16286;
    --def_color: #689d6a; /* class name */
    --property_color: #458588; /* python properties */
    --builtin_color: #689d6a;
    --comment_color: #9f9f9f;
    --comment_color_2: #458588; /* equals sign */
    --operator_color: #a221f2;
    --font_color: #3c3836; /* general font */
    --variable2_color: #b16286; /*self keyworda */
    --box_color: #fffdee66;
  }

  /* Add the following style for headers with background color */
  h1,
  .h1 {
    font-family: "Trebuchet MS", sans-serif;
    font-size: 2em !important;
    letter-spacing: 1px;
    color: var(--header1_color);
    border-bottom: 3px solid var(--header1_color);
    background-color: #000080;
    padding: 0.5em;
    color: #ffff00 !important;
  }

  h2,
  .h2 {
    font-family: "Trebuchet MS";
    font-size: 1.7em !important;
    color: var(--header2_color);
    background-color: #000080;
    padding: 0.5em;
    color: #ffff00 !important;
  }

  h3,
  .h3 {
    font-family: "Trebuchet MS";
    font-size: 1.4em !important;
    color: var(--header3_color);
    background-color: #000080;
    padding: 0.5em;
    color: #ffff00 !important;
  }

  /* Rest of your existing styles... */

  body[data-jp-theme-light="true"] .jp-Notebook .CodeMirror.cm-s-jupyter {
    background-color: var(--box_color) !important;
  }

  div.input_area {
    background-color: var(--box_color) !important;
  }
</style>

"""

# Apply custom CSS
HTML(custom_css)

In [ ]:
%%capture
!pip install bluecast #--no-index --find-links=file:/kaggle/input/bluecast/bluecast-0.90-py3-none-any.whl

<a id="toc"></a>

<a href="#toc" style="background-color: #E1B12D; color: #ffffff; padding: 7px 10px; text-decoration: none; border-radius: 50px;">Back to top</a><a id="toc"></a>

<a id="1.2"></a>

# Load the data

In [ ]:
from category_encoders import (
    GLMMEncoder,
    LeaveOneOutEncoder,
    OneHotEncoder,
    OrdinalEncoder,
    TargetEncoder,
    WOEEncoder,
)

from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler

import numpy as np
import pandas as pd
import re
from typing import Optional, Tuple, Union

import matplotlib.pyplot as plt
from xgboost import plot_tree

import warnings
warnings.filterwarnings("ignore")


from bluecast.blueprints.cast import BlueCast
from bluecast.blueprints.cast_cv import BlueCastCV
from bluecast.config.training_config import TrainingConfig, XgboostTuneParamsConfig
from bluecast.preprocessing.custom import CustomPreprocessing
from bluecast.general_utils.general_utils import save_to_production, load_for_production

from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.preprocessing import PowerTransformer, LabelEncoder

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s4e2/train.csv')
train["source"] = 0

df_orig = pd.read_csv('/kaggle/input/obesity-or-cvd-risk-classifyregressorcluster/ObesityDataSet.csv')
df_orig["source"] = 1

test = pd.read_csv('/kaggle/input/playground-series-s4e2/test.csv')
test["source"] = 1
submission = pd.read_csv('/kaggle/input/playground-series-s4e2/sample_submission.csv')

cols = df_orig.columns
train = pd.concat((train, df_orig), axis=0).reset_index(drop=True)

print('The dimension of the train dataset is:', train.shape)
print('The dimension of the test dataset is:', test.shape)

In [ ]:
train

In [ ]:
test

In [ ]:
train.info()

In [ ]:
train.describe()

<a id="toc"></a>

<a href="#toc" style="background-color: #E1B12D; color: #ffffff; padding: 7px 10px; text-decoration: none; border-radius: 50px;">Back to top</a><a id="toc"></a>

<a id="1.2"></a>

# Feature engineering

In this section we will build features using various techniques.

In [ ]:
target = "NObeyesdad"

In [ ]:
# from here: https://www.kaggle.com/code/natapelysynka/obesity-risk-multiclassifier-feature-eng-xgb0-97
def transform_feats(data):
    data['Gender_binary'] = data['Gender'].map({'Male': 1, 'Female': 0}).astype(int)
    data['family_history_with_overweight_binary'] = data['family_history_with_overweight'].map({'yes': 1, 'no': 0}).astype(int)
    data['SMOKE_binary'] = data['SMOKE'].map({'yes': 1, 'no': 0}).astype(int)
    
    # taken from: https://www.kaggle.com/code/ravi20076/playgrounds4e02-extraftre-models
    data["CAEC"] = data["CAEC"].map({"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3}).astype(np.uint8)
    data['SCC']  = np.where(data["SCC"] == "no", 1,0).astype(np.uint8)
    data["CALC"] = data["CALC"].map({"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 2}).astype(np.uint8)
    return data

def feature_engineering(data):
    # BMI
    data['BMI'] = data['Weight'] / (data['Height'] ** 2)
    # Activity
    data['Activity'] = data['FAF'] * data['TUE']
    # Age group
    data['Age_Group'] = pd.cut(data['Age'], bins=[0, 18, 30, 45, float('inf')], labels=[0, 1, 2, 3])
    data['Age_Group'] = data['Age_Group'].astype(int)
    # Height group
    data['Height_Group'] = pd.cut(data['Height'], bins=[0, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, float('inf')], labels=[0, 1, 2, 3, 4, 5, 6])
    data['Height_Group'] = data['Height_Group'].astype(int)
    #Risk score
    data['Risk factor'] = (data['BMI'] + data['Age_Group']) * (data["family_history_with_overweight_binary"] + data["SMOKE_binary"])
    
    # taken from: https://www.kaggle.com/code/ravi20076/playgrounds4e02-extraftre-models
    data["BMIbyNCP"] = np.log1p(data["BMI"]) - np.log1p(data["NCP"])
    data["BMIFAF"] = (data["BMI"] * data["FAF"])/ 25.0
    data["FAFmTUE"] = data["FAF"] - data["TUE"]
    data["FCVCpNCP"] = data['FCVC'] * data['NCP']
    data['TechUse'] = np.log1p(data['TUE']) - np.log1p(data['Age'])
    return data

In [ ]:
train = transform_feats(train)
test = transform_feats(test)

In [ ]:
train.info()

In [ ]:
def get_group_zscores(df, group_cols, agg_col):
    df_gr = df.groupby(group_cols).agg({agg_col: ["mean", "std"]}).droplevel(0, axis=1).reset_index()
    df_gr["mean"] = df_gr["mean"].fillna(df_gr["mean"].mean())
    df_gr["std"] = df_gr["std"].fillna(df_gr["std"].mean())
    
    identifier = "_".join(group_cols) + "_" + agg_col
    df_gr = df_gr.rename(
        columns = {
            "mean": f"{identifier}_mean",
            "std": f"{identifier}_std"
        }
    )
    df = df.merge(df_gr, on=group_cols, how="left")
    df[f"{identifier}_zscore"] = (df[f"{identifier}_mean"] - df[agg_col]) / df[f"{identifier}_std"]
    return df

In [ ]:
train[target].value_counts()

<a id="toc"></a>

<a href="#toc" style="background-color: #E1B12D; color: #ffffff; padding: 7px 10px; text-decoration: none; border-radius: 50px;">Back to top</a><a id="toc"></a>

<a id="1.2"></a>

# EDA with BlueCast

Here you can get an overview of the data and its distribution.

In [ ]:
from bluecast.eda.analyse import (
    plot_pie_chart,
    plot_count_pairs,
    bi_variate_plots,
    correlation_heatmap,
    correlation_to_target,
    plot_pca,
    plot_pca_cumulative_variance,
    plot_theil_u_heatmap,
    plot_tsne,
    univariate_plots,
    check_unique_values,
    plot_null_percentage,
    mutual_info_to_target
)

from bluecast.preprocessing.feature_types import FeatureTypeDetector

## Feature type detection

In [ ]:
ignore_cols = []

feat_type_detector = FeatureTypeDetector()
train_data = feat_type_detector.fit_transform_feature_types(train.drop(ignore_cols, axis=1))

len(feat_type_detector.num_columns)

In [ ]:
le = LabelEncoder()

# Target class distribution

In [ ]:
plot_pie_chart(
        train_data,
        target,
    )

## Univariate plots

In [ ]:
univariate_plots(
        train_data.loc[
            :, feat_type_detector.num_columns
        ],
    )

## Bivariate plots

In [ ]:
encoded_target_train = train.copy()
encoded_target_train[target] = le.fit_transform(encoded_target_train[target])

bi_variate_plots(
        encoded_target_train.loc[
            :, feat_type_detector.num_columns + [target]
        ],
        target,
    )

In some features we can see clear differences between the classes.

# Count pairs

In [ ]:
plot_count_pairs(
    train,
    test,
    cat_cols=train_data.loc[:, feat_type_detector.cat_columns],
      )

## Correlation to target

In [ ]:
# show correlation to target
correlation_to_target(
    encoded_target_train.loc[:, feat_type_detector.num_columns + [target]],
      target,
      )

Weigh, hight and age show good correlation coeffcients. They have a linear relationship to the target.

## Correlation heatmap

In [ ]:
correlation_heatmap(train_data.loc[
            :, feat_type_detector.num_columns])

## Mutual information score

In [ ]:
# show mutual information of categorical features to target
# features are expected to be numerical format
extra_params = {"random_state": 30}
mutual_info_to_target(encoded_target_train.loc[:, feat_type_detector.num_columns + [target]].fillna(0), target, class_problem="multiclass", **extra_params)

Weight, age and height have very high MI scores!

## Dimensionality reduction using PCA

In [ ]:
# show feature space after principal component analysis
plot_pca(train_data.loc[
            :, feat_type_detector.num_columns + [target]
        ].fillna(0), target)

It looks like linear models can be promising here. Most classes can be linearly separated very well.

## Show cumulative variance of principal components

In [ ]:
## show how many components are needed to explain certain variance
plot_pca_cumulative_variance(
    train_data.loc[:, feat_type_detector.num_columns].fillna(0),
    "target",
    n_components=13
    )

## Dimensionality reduction using t-SNE

In [ ]:
# show feature space after t-SNE
plot_tsne(train_data.loc[
            :, feat_type_detector.num_columns + [target]
        ].fillna(0), target, perplexity=1000, random_state=0)

## Map of associations between categorical features

In [ ]:
# show a heatmap of assocations between categorical variables
if len(feat_type_detector.cat_columns) > 0:
    theil_matrix = plot_theil_u_heatmap(train_data, feat_type_detector.cat_columns)
    theil_matrix

## Missing values

In [ ]:
# plot the percentage of Nulls for all features
if train_data.loc[:, feat_type_detector.num_columns].isna().sum().sum() > 0:
    plot_null_percentage(
       train_data.loc[:, feat_type_detector.num_columns],
        )
else:
    print("This dataset does not have any missing values")

## Do we have columns with high cardinality?

In [ ]:
# detect columns with a very high share of unique values
many_unique_cols = check_unique_values(train_data, feat_type_detector.cat_columns, threshold=0.90)
many_unique_cols

# Leakage detection

With big data and complex pipelines data leakage can easily sneak in.
To detect leakage BlueCast offers two functions:

In [ ]:
from bluecast.eda.data_leakage_checks import (
    detect_categorical_leakage,
    detect_leakage_via_correlation,
)

## Leakage detection for numerical columns

In [ ]:
# Detect leakage of numeric columns based on correlation
numresult = detect_leakage_via_correlation(
        encoded_target_train.loc[:, feat_type_detector.num_columns + [target]], target, threshold=0.9 # target column is part of detected numerical columns here
    )
numresult

## Leakage detection for categorical columns

In [ ]:
# Detect leakage of categorical columns based on Theil's U
result = detect_categorical_leakage(
        encoded_target_train.loc[:, feat_type_detector.cat_columns + [target]], target, threshold=0.9
    )
result

This could also be a false positive when these features are contant. We might neeed to drop these.

# Check for data drift

In [ ]:
from bluecast.monitoring.data_monitoring import DataDrift

In [ ]:
data_drift = DataDrift()

## Data drift for numerical columns

In [ ]:
data_drift.kolmogorov_smirnov_test(train, test)
data_drift.kolmogorov_smirnov_flags

## Data drift for categorical columns

In [ ]:
data_drift.population_stability_index(train.drop(target, axis=1), test)
data_drift.population_stability_index_flags

In [ ]:
for col, data_drift_flag in data_drift.kolmogorov_smirnov_flags.items():
    data_drift.qqplot_two_samples(train[col], test[col], x_label=f"{col} from train", y_label=f"{col} from test")

Except from id column no data drift could be detected.

<a id="toc"></a>

<a href="#toc" style="background-color: #E1B12D; color: #ffffff; padding: 7px 10px; text-decoration: none; border-radius: 50px;">Back to top</a><a id="toc"></a>

<a id="1.2"></a>

# Building the pipeline in a few lines of code

In [ ]:
from bluecast.blueprints.cast import BlueCast
from bluecast.preprocessing.custom import CustomPreprocessing

from sklearn.ensemble import IsolationForest


# add custom last mile computation
class CustomInFoldPreprocessing(CustomPreprocessing):
    def __init__(self):
        self.if_detector = None
    # Please note: The base class enforces that the fit_transform method is implemented
    def fit_transform(
        self, df: pd.DataFrame, target: pd.Series
    ) -> Tuple[pd.DataFrame, pd.Series]:
        train = df.copy()
        train = feature_engineering(train)
        train = get_group_zscores(train, ["Height_Group", "Age_Group"], "BMI")
        train = get_group_zscores(train, ["Height_Group", "Age_Group"], "Risk factor")
        train = get_group_zscores(train, ["Gender_binary", "SMOKE_binary", "family_history_with_overweight_binary"], "BMI")
        train = get_group_zscores(train, ["Gender_binary", "SMOKE_binary", "family_history_with_overweight_binary"], "Risk factor")
        train = train.drop(["source", "Age_Group", "Height_Group"], axis=1)
        train_target = target.copy().astype(float)
        train = train.replace([np.inf, -np.inf], 0)
        
        # add outlier scores as feature
        self.if_detector = IsolationForest(random_state=0)
        self.if_detector.fit(train.fillna(0))
        train["isolation_forest_scores"] = self.if_detector.predict(train.fillna(0))
        return train, train_target

    # Please note: The base class enforces that the fit_transform method is implemented
    def transform(
        self,
        df: pd.DataFrame,
        target: Optional[pd.Series] = None,
        predicton_mode: bool = False,
    ) -> Tuple[pd.DataFrame, Optional[pd.Series]]:
        train = df.copy()
        
        if isinstance(target, pd.Series) or isinstance(target, np.ndarray):
            train["target"] = target.copy()
            train["target"] = train["target"].astype(float)
            train = train.loc[train["source"] == 0].reset_index(drop=True) # no original data
            target = train.pop("target")
            target = pd.Series(target).astype(float)
        
        train = feature_engineering(train)
        train = get_group_zscores(train, ["Height_Group", "Age_Group"], "BMI")
        train = get_group_zscores(train, ["Height_Group", "Age_Group"], "Risk factor")
        train = get_group_zscores(train, ["Gender_binary", "SMOKE_binary", "family_history_with_overweight_binary"], "BMI")
        train = get_group_zscores(train, ["Gender_binary", "SMOKE_binary", "family_history_with_overweight_binary"], "Risk factor")
        train = train.drop(["source", "Age_Group", "Height_Group"], axis=1)
        train = train.replace([np.inf, -np.inf], 0)
        
        train["isolation_forest_scores"] = self.if_detector.predict(train.fillna(0))
        return train, target

custom_preprocessor = CustomInFoldPreprocessing()

In [ ]:
from bluecast.config.training_config import TrainingConfig, XgboostTuneParamsConfig

# We give more depth
xgboost_param_config = XgboostTuneParamsConfig()
xgboost_param_config.steps_max = 1000
xgboost_param_config.max_depth_max = 7

# Create a custom training config and adjust general training parameters
train_config = TrainingConfig()
train_config.global_random_state = 600
train_config.hypertuning_cv_folds = 1
train_config.hyperparameter_tuning_rounds = 500
train_config.hyperparameter_tuning_max_runtime_secs = 60 * 60 * 2
#train_config.enable_grid_search_fine_tuning = True # enable param refinement
train_config.use_full_data_for_final_model = True
#train_config.precise_cv_tuning = True
train_config.gridsearch_nb_parameters_per_grid = 5
#train_config.cat_encoding_via_ml_algorithm = True
#train_config.calculate_shap_values = False

skf = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=1987)

In [ ]:
automl = BlueCastCV(
        class_problem="multiclass", # also multiclass is possible
        #stratifier=skf,
        conf_training=train_config,
        conf_xgboost=xgboost_param_config,
        #custom_in_fold_preprocessor=custom_preprocessor,
        #custom_preprocessor=custom_preprocessor,
        #ml_model=custom_model_tab,
        )

In [ ]:
try:
    automl.fit_eval(train.copy(), target_col=target)
except Exception as e:
    print(e)

<a id="toc"></a>

<a href="#toc" style="background-color: #E1B12D; color: #ffffff; padding: 7px 10px; text-decoration: none; border-radius: 50px;">Back to top</a><a id="toc"></a>

<a id="1.2"></a>

# Plot decision trees

In [ ]:
for model in automl.bluecast_models:
    plot_tree(model.ml_model.model)
    fig = plt.gcf()
    fig.set_size_inches(150, 80)
    plt.show()

# Predict on new data

In [ ]:
probs, classes = automl.predict(test)

# Accessing the inbuilt experiment tracker

BlueCast also keep track of your experiments! Let's see how we can access them.

In [ ]:
# access the experiment tracker if needed
tracker = automl.experiment_tracker
tracker

In [ ]:
# see all stored information as a Pandas DataFrame
tracker_df = tracker.retrieve_results_as_df()
tracker_df

In [ ]:
tracker_df.info()

Let us try to find out what the most important hyperparameters and settings have been so far

<a id="toc"></a>

<a href="#toc" style="background-color: #E1B12D; color: #ffffff; padding: 7px 10px; text-decoration: none; border-radius: 50px;">Back to top</a><a id="toc"></a>

<a id="1.2"></a>

## Understand most impactful parameters across all hyperparameters tests and model trainings

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import shap 

cols = [
    "shuffle_during_training",
    "global_random_state",
    "early_stopping_rounds",
    "autotune_model",
    "enable_feature_selection",
    "train_split_stratify",
    "use_full_data_for_final_model",
    "eta",
    "max_depth",
    "alpha",
    "lambda",
    "gamma",
    "max_leaves",
    "subsample",
    "colsample_bytree",
    "colsample_bylevel"
]

regr = RandomForestRegressor(max_depth=4, random_state=0)

tracker_df = tracker_df.loc[tracker_df["score_category"] == "oof_score"]

experiment_feats_df, experiment_feats_target = tracker_df.loc[:, cols], tracker_df.loc[:, "eval_scores"]

regr.fit(experiment_feats_df.fillna(0), experiment_feats_target.fillna(99))

explainer = shap.TreeExplainer(regr)


shap_values = explainer.shap_values(experiment_feats_df)
shap.summary_plot(shap_values, experiment_feats_df)

## Don't lose your progress!

BlueCast offers simple utilities to save and load your pipeline (including the tracker)

In [ ]:
# save pipeline including tracker
save_to_production(automl, "/kaggle/working/", "bluecast_cv_pipeline")

# in production or for further experiments this can be loaded again
automl_loaded = load_for_production("/kaggle/working/", "bluecast_cv_pipeline")

# Submission time

In [ ]:
classes

In [ ]:
submission[target] = classes
submission.to_csv('submission.csv', index=False)

In [ ]:
submission

<a id="toc"></a>

<a href="#toc" style="background-color: #E1B12D; color: #ffffff; padding: 7px 10px; text-decoration: none; border-radius: 50px;">Back to top</a><a id="toc"></a>

<a id="1.2"></a>